## Task
Compute time intervals between adjacent commits in a chosen Git repository using **GitPython**, then visualize those intervals as a **Plotly** bar chart (newest→oldest commit pairs).

In [49]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\Rainybyte\AppData\Roaming\JetBrains\DataSpell2025.3\projects\workspace\.venv\Scripts\python.exe
3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]


In [50]:
import sys

# Install into the *current kernel* environment
!"{sys.executable}" -m pip install -U gitpython plotly pandas nbformat


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Rainybyte\AppData\Roaming\JetBrains\DataSpell2025.3\projects\workspace\.venv\Scripts\python.exe -m pip install --upgrade pip


In [51]:
from pathlib import Path
from git import Repo
import pandas as pd
import plotly.express as px

In [52]:
REPO_PATH = Path(r"C:\Users\Rainybyte\Documents\Obsidian\Research Projects")

In [53]:
if REPO_PATH is None:
    REPO_PATH = Path.cwd()  # change to Path(r"C:\path\to\repo") if needed

repo = Repo(str(REPO_PATH), search_parent_directories=True)
commits = list(repo.iter_commits())  # default: HEAD history (newest -> oldest)

rows = []
for i in range(len(commits) - 1):
    newer = commits[i]
    older = commits[i + 1]
    dt_newer = pd.to_datetime(newer.committed_datetime)
    dt_older = pd.to_datetime(older.committed_datetime)
    delta = (dt_newer - dt_older).total_seconds()

    rows.append(
        {
            "pair_index": i,
            "newer_hash": newer.hexsha[:8],
            "older_hash": older.hexsha[:8],
            "newer_time": dt_newer,
            "older_time": dt_older,
            "interval_seconds": delta,
            "interval_minutes": delta / 60.0,
            "interval_hours": delta / 3600.0,
            "label": f"{newer.hexsha[:8]} → {older.hexsha[:8]}",
        }
    )

df_intervals = pd.DataFrame(rows)

df_intervals.head()

,pair_index,newer_hash,older_hash,newer_time,older_time,interval_seconds,interval_minutes,interval_hours,label
0,0,b8365f15,ae6aea65,2026-02-16 18:58:34+01:00,2026-02-16 16:51:53+01:00,7601.0,126.683333,2.111389,b8365f15 → ae6aea65
1,1,ae6aea65,2aadf4ee,2026-02-16 16:51:53+01:00,2026-02-16 16:08:35+01:00,2598.0,43.300000,0.721667,ae6aea65 → 2aadf4ee
2,2,2aadf4ee,0ed3d030,2026-02-16 16:08:35+01:00,2026-02-16 15:20:41+01:00,2874.0,47.900000,0.798333,2aadf4ee → 0ed3d030
3,3,0ed3d030,0ad4fa53,2026-02-16 15:20:41+01:00,2026-02-16 14:45:49+01:00,2092.0,34.866667,0.581111,0ed3d030 → 0ad4fa53
4,4,0ad4fa53,b9b318fc,2026-02-16 14:45:49+01:00,2026-02-16 13:59:55+01:00,2754.0,45.900000,0.765000,0ad4fa53 → b9b318fc


In [60]:
fig = px.histogram(
    df_intervals,
    x="interval_minutes",
    range_x=[0.1, 600],
    nbins=2000,  # Adjust the number of bins as needed
    hover_data={
        "newer_time": True,
        "older_time": True,
        "interval_seconds": ":.0f",
        "interval_minutes": ":.2f",
        "interval_hours": ":.3f",
    },
    title=f"Histogram of Commit Intervals ({repo.working_tree_dir})",
    labels={"interval_minutes": "Interval (minutes)"},
)

fig.update_layout(
    xaxis_title="Commit Interval (minutes)",
    yaxis_title="Frequency",
    bargap=0.1,
    height=600,
)
fig.show()